# Chapter 12: Agentic Training — Teaching Agents to Self-Improve

> **Part 3: Reasoning** | Guardian Angel System (GAS) Book

---

## 🎯 Chapter Goal

In this chapter we explore how agents can **learn from their own experience** — without requiring human-labeled datasets for every new skill. We cover three landmark techniques:

| Technique | Core Idea | Key Benefit |
|-----------|-----------|-------------|
| **STaR** (Self-Taught Reasoner) | Generate rationales → filter correct ones → fine-tune | Bootstraps reasoning from unlabeled data |
| **Reflexion** | Verbal reinforcement via self-reflection stored in memory | Improves without gradient updates |
| **ReAct** | Interleave reasoning traces with tool actions | Grounds reasoning in real-world feedback |

By the end of this chapter you will:
- Implement a **STaR data-collection loop** for medical reasoning
- Build a **Reflexion agent** that improves over 3 iterations
- Understand how these techniques power the **GAS Advisor Agent**

---

## 📖 Theory Recap

### STaR — Self-Taught Reasoner (Zelikman et al., 2022)

STaR bootstraps reasoning ability from a small seed of labeled examples:

```
Loop:
  1. For each unlabeled question Q:
       Generate N reasoning chains → answer pairs
  2. Filter: keep only chains where final answer == correct answer
  3. Fine-tune the model on (Q, correct_chain) pairs
  4. Repeat with the improved model
```

**Rationalization trick**: For questions the model gets wrong, provide the correct answer as a hint and ask it to rationalize — then add those to training data too.

---

### Reflexion — Verbal Reinforcement Learning (Shinn et al., 2023)

Reflexion replaces gradient updates with **natural language reflection**:

```
Loop:
  1. Agent attempts task → gets outcome (success/failure + feedback)
  2. Agent writes a reflection: "I failed because X. Next time I will Y."
  3. Reflection is stored in episodic memory
  4. On next attempt, reflection is prepended to the context
  5. Agent retries with improved strategy
```

Key insight: **language models already know how to reflect** — they just need a structured opportunity to do so.

---

### ReAct — Reasoning + Acting (Yao et al., 2022)

ReAct interleaves **Thought → Action → Observation** traces:

```
Thought: I need to check the patient's last 3 glucose readings.
Action: query_glucose_history(patient_id=42, n=3)
Observation: [280, 310, 265] mg/dL over last 6 hours
Thought: All readings are high. I should check if insulin was administered.
Action: query_insulin_log(patient_id=42)
Observation: Last dose 2 hours ago, 8 units rapid-acting
Thought: Insulin was given but glucose remains elevated. Recommend re-check in 30 min.
Answer: Monitor closely, re-check glucose in 30 minutes.
```

ReAct dramatically reduces hallucination by grounding reasoning in tool observations.

---

## 🔨 Build It

### Setup

In [ ]:
# pip install openai matplotlib numpy
# Uncomment to install:
# import subprocess, sys
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openai', 'matplotlib', 'numpy', '-q'])

In [ ]:
import os
import json
import random
import time
import copy
from typing import List, Dict, Optional, Tuple
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Test Mode ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))
from llm_utils import llm, TEST_MODE, BACKEND, MODEL
print(f"Python path: {os.sys.executable}")

### Part 1: STaR — Self-Taught Reasoner Loop

We implement the STaR data-collection loop for a medical reasoning task. The agent generates multiple reasoning chains, filters the correct ones, and collects them as training data.

In [ ]:
# ── STaR Implementation ────────────────────────────────────────────────────────

MEDICAL_QUESTION = (
    "Patient has Type 1 diabetes, last glucose reading 280 mg/dL, "
    "took 8 units rapid-acting insulin 2 hours ago. "
    "What should happen next?"
)

CORRECT_ANSWER_KEYWORDS = ["monitor", "recheck", "30 min", "30min", "check again", "observe"]

# Mock reasoning chains for TEST_MODE
MOCK_CHAINS = [
    {
        "chain": (
            "Step 1: Glucose is 280 mg/dL — significantly above target range (70-180 mg/dL).\n"
            "Step 2: Insulin was administered 2 hours ago. Rapid-acting insulin peaks at 1-3 hours.\n"
            "Step 3: The insulin may still be working. Giving more insulin now risks hypoglycemia.\n"
            "Step 4: Best action is to monitor and recheck glucose in 30 minutes.\n"
            "Answer: Monitor closely and recheck glucose in 30 minutes."
        ),
        "answer": "Monitor closely and recheck glucose in 30 minutes.",
        "correct": True,
    },
    {
        "chain": (
            "Step 1: Glucose 280 mg/dL is high.\n"
            "Step 2: Patient needs more insulin immediately.\n"
            "Step 3: Administer correction dose now.\n"
            "Answer: Give additional insulin correction dose immediately."
        ),
        "answer": "Give additional insulin correction dose immediately.",
        "correct": False,  # Dangerous — ignores recent dose
    },
    {
        "chain": (
            "Step 1: 280 mg/dL is above normal range.\n"
            "Step 2: Insulin was given 2 hours ago — it is still active.\n"
            "Step 3: Wait for insulin to take effect. Check again in 30 min.\n"
            "Step 4: If still elevated after 30 min, consult care team.\n"
            "Answer: Check again in 30 min; consult care team if still elevated."
        ),
        "answer": "Check again in 30 min; consult care team if still elevated.",
        "correct": True,
    },
    {
        "chain": (
            "Step 1: High glucose detected.\n"
            "Step 2: Patient should eat less carbohydrates.\n"
            "Answer: Reduce carbohydrate intake."
        ),
        "answer": "Reduce carbohydrate intake.",
        "correct": False,  # Irrelevant to immediate management
    },
    {
        "chain": (
            "Step 1: Glucose 280 mg/dL — hyperglycemia confirmed.\n"
            "Step 2: Recent insulin dose (2h ago) is still in active window.\n"
            "Step 3: Risk of stacking insulin doses is hypoglycemia.\n"
            "Step 4: Safe protocol: monitor and recheck in 30 minutes.\n"
            "Step 5: Ensure patient is hydrated.\n"
            "Answer: Monitor, ensure hydration, recheck glucose in 30 minutes."
        ),
        "answer": "Monitor, ensure hydration, recheck glucose in 30 minutes.",
        "correct": True,
    },
]


def call_llm(prompt: str, model: str = None, temperature: float = 0.8) -> str:
    """Call LLM — uses llm_utils backend (Ollama/OpenAI/TEST_MODE)."""
    if TEST_MODE:
        return random.choice(MOCK_CHAINS)["chain"]
    return llm.chat([{"role": "user", "content": prompt}], temperature=temperature)


def is_correct(chain: str, correct_keywords: List[str]) -> bool:
    """Simple correctness check: does the chain contain any correct-answer keywords?"""
    chain_lower = chain.lower()
    return any(kw.lower() in chain_lower for kw in correct_keywords)


def star_loop(
    question: str,
    n_chains: int = 5,
    correct_keywords: Optional[List[str]] = None,
) -> Dict:
    """
    STaR data-collection loop.
    
    Args:
        question: The medical question to reason about.
        n_chains: Number of reasoning chains to generate.
        correct_keywords: Keywords that indicate a correct answer.
    
    Returns:
        Dict with 'all_chains', 'correct_chains', 'training_data', 'accuracy'.
    """
    if correct_keywords is None:
        correct_keywords = CORRECT_ANSWER_KEYWORDS
    
    prompt_template = (
        "You are a clinical decision support AI. Reason step-by-step about the following "
        "medical question. Show your reasoning chain clearly, then give a final Answer.\n\n"
        f"Question: {question}\n\n"
        "Reasoning chain:"
    )
    
    all_chains = []
    correct_chains = []
    
    print(f"🔄 STaR Loop: Generating {n_chains} reasoning chains...")
    print(f"   Question: {question[:80]}...\n")
    
    for i in range(n_chains):
        if TEST_MODE:
            # Use pre-defined mock chains for reproducibility
            chain_data = MOCK_CHAINS[i % len(MOCK_CHAINS)]
            chain_text = chain_data["chain"]
        else:
            chain_text = call_llm(prompt_template, temperature=0.8)
        
        correct = is_correct(chain_text, correct_keywords)
        
        chain_record = {
            "id": i + 1,
            "chain": chain_text,
            "correct": correct,
        }
        all_chains.append(chain_record)
        
        status = "✅ CORRECT" if correct else "❌ INCORRECT"
        print(f"  Chain {i+1}: {status}")
        # Show first line of chain
        first_line = chain_text.split("\n")[0][:70]
        print(f"    → {first_line}")
        
        if correct:
            correct_chains.append(chain_record)
    
    # Build training data: (question, correct_chain) pairs
    training_data = [
        {"input": question, "reasoning": c["chain"]}
        for c in correct_chains
    ]
    
    accuracy = len(correct_chains) / n_chains
    
    print(f"\n📊 Results:")
    print(f"   Total chains generated: {n_chains}")
    print(f"   Correct chains: {len(correct_chains)}")
    print(f"   Accuracy: {accuracy:.0%}")
    print(f"   Training examples collected: {len(training_data)}")
    
    return {
        "all_chains": all_chains,
        "correct_chains": correct_chains,
        "training_data": training_data,
        "accuracy": accuracy,
    }


# Run STaR loop
star_results = star_loop(MEDICAL_QUESTION, n_chains=5)

print("\n📋 Sample training example:")
if star_results["training_data"]:
    sample = star_results["training_data"][0]
    print(f"  Input: {sample['input'][:60]}...")
    print(f"  Reasoning (first 2 lines):")
    for line in sample["reasoning"].split("\n")[:2]:
        print(f"    {line}")

### Part 2: Reflexion — Verbal Reinforcement Learning

The Reflexion agent attempts a task, receives feedback, writes a reflection, and improves on the next attempt — all without gradient updates.

In [ ]:
# ── Reflexion Implementation ───────────────────────────────────────────────────

# Mock responses for TEST_MODE — showing progressive improvement
MOCK_ATTEMPTS = [
    # Iteration 1: Poor attempt
    (
        "Recommendation: Increase basal insulin by 20% to address morning hyperglycemia.",
        "POOR",
        "The recommendation is too aggressive. A 20% basal increase without "
        "considering dawn phenomenon timing is dangerous. Missing: timing analysis, "
        "patient-specific factors, gradual titration protocol.",
    ),
    # Iteration 2: Better attempt
    (
        "Recommendation: Check if morning highs occur consistently. If dawn phenomenon, "
        "consider increasing basal insulin by 10% and recheck in 3 days.",
        "ACCEPTABLE",
        "Better — includes pattern check and gradual titration. Missing: "
        "specific timing of basal adjustment (should be evening dose for dawn phenomenon), "
        "and patient education component.",
    ),
    # Iteration 3: Good attempt
    (
        "Recommendation: Patient shows dawn phenomenon pattern (HbA1c 9.2%, morning highs). "
        "Action plan: (1) Increase evening long-acting insulin by 10%, "
        "(2) Recheck fasting glucose for 3 days, "
        "(3) Educate patient on dawn phenomenon, "
        "(4) Follow up in 1 week. Safety: monitor for nocturnal hypoglycemia.",
        "GOOD",
        "Excellent — addresses root cause, includes safety monitoring, patient education, "
        "and follow-up plan. Meets clinical guidelines.",
    ),
]

MOCK_REFLECTIONS = [
    # Reflection after iteration 1
    (
        "I failed because I jumped to a solution without analyzing the pattern. "
        "A 20% increase is too aggressive and ignores the dawn phenomenon mechanism. "
        "Next time I will: (1) first identify the pattern (dawn phenomenon vs. "
        "insufficient basal), (2) recommend gradual titration (10% max), "
        "(3) specify which insulin to adjust based on timing."
    ),
    # Reflection after iteration 2
    (
        "I improved but still missed key details. I correctly identified the need for "
        "pattern checking and gradual titration. However, I did not specify that for "
        "dawn phenomenon, the EVENING long-acting dose should be adjusted, not the morning. "
        "I also forgot patient education and safety monitoring for nocturnal hypoglycemia. "
        "Next time I will include: timing-specific insulin adjustment, patient education, "
        "and explicit safety monitoring."
    ),
]


class ReflexionAgent:
    """
    A Reflexion agent that improves through verbal self-reflection.
    
    The agent maintains an episodic memory of reflections. On each retry,
    past reflections are prepended to the context, allowing the agent to
    learn from its mistakes without gradient updates.
    """
    
    def __init__(self, name: str = "GAS-Advisor-Reflexion"):
        self.name = name
        self.reflection_memory: List[str] = []
        self.attempt_history: List[Dict] = []
        self.iteration = 0
    
    def _build_context(self, task: str) -> str:
        """Build the full context including past reflections."""
        context = f"Task: {task}\n\n"
        if self.reflection_memory:
            context += "Past reflections (learn from these):\n"
            for i, reflection in enumerate(self.reflection_memory, 1):
                context += f"  Reflection {i}: {reflection}\n"
            context += "\n"
        context += "Your recommendation:"
        return context
    
    def attempt(self, task: str) -> str:
        """Make an attempt at the task."""
        context = self._build_context(task)
        
        if TEST_MODE:
            response = MOCK_ATTEMPTS[min(self.iteration, len(MOCK_ATTEMPTS) - 1)][0]
        else:
            prompt = (
                "You are a clinical AI advisor for a diabetic patient supervision system.\n\n"
                + context
            )
            response = call_llm(prompt, temperature=0.3)
        
        self.attempt_history.append({
            "iteration": self.iteration + 1,
            "attempt": response,
            "context_length": len(context),
        })
        return response
    
    def reflect(self, outcome: str, feedback: str) -> str:
        """Write a reflection based on outcome and feedback."""
        if TEST_MODE:
            reflection = MOCK_REFLECTIONS[min(self.iteration, len(MOCK_REFLECTIONS) - 1)]
        else:
            last_attempt = self.attempt_history[-1]["attempt"] if self.attempt_history else ""
            prompt = (
                f"You attempted a clinical task and received this feedback:\n"
                f"Your attempt: {last_attempt}\n"
                f"Outcome: {outcome}\n"
                f"Feedback: {feedback}\n\n"
                "Write a concise reflection (2-3 sentences) on what went wrong "
                "and exactly what you will do differently next time:"
            )
            reflection = call_llm(prompt, temperature=0.3)
        
        self.reflection_memory.append(reflection)
        self.iteration += 1
        return reflection
    
    def get_score(self) -> float:
        """Return a quality score for the current iteration (for plotting)."""
        scores = {"POOR": 0.25, "ACCEPTABLE": 0.60, "GOOD": 0.90}
        if TEST_MODE and self.iteration < len(MOCK_ATTEMPTS):
            return scores.get(MOCK_ATTEMPTS[self.iteration][1], 0.5)
        return 0.5


# ── Run Reflexion Loop ─────────────────────────────────────────────────────────
REFLEXION_TASK = (
    "Recommend an insulin adjustment strategy for a patient with HbA1c 9.2% "
    "and frequent morning hyperglycemia (fasting glucose consistently 180-220 mg/dL). "
    "Patient is on basal-bolus insulin therapy."
)

agent = ReflexionAgent(name="GAS-Advisor")
iteration_scores = []

print("=" * 70)
print("🔄 REFLEXION LOOP — 3 Iterations")
print("=" * 70)
print(f"\nTask: {REFLEXION_TASK[:80]}...\n")

for iteration in range(3):
    print(f"\n{'─' * 60}")
    print(f"📍 ITERATION {iteration + 1}")
    print(f"{'─' * 60}")
    
    # Score before attempt (for plotting)
    score = agent.get_score()
    iteration_scores.append(score)
    
    # Attempt
    print(f"\n🤖 Agent Attempt:")
    response = agent.attempt(REFLEXION_TASK)
    print(f"   {response}")
    
    # Feedback (in real system, this comes from Safety Critic or outcome)
    if TEST_MODE:
        outcome, feedback = MOCK_ATTEMPTS[iteration][1], MOCK_ATTEMPTS[iteration][2]
    else:
        outcome = "NEEDS_IMPROVEMENT"
        feedback = "Please be more specific about timing and safety monitoring."
    
    print(f"\n📋 Feedback: [{outcome}] {feedback[:80]}...")
    
    # Reflect (only for first 2 iterations)
    if iteration < 2:
        reflection = agent.reflect(outcome, feedback)
        print(f"\n💭 Reflection stored:")
        print(f"   {reflection[:120]}...")
        print(f"   (Memory now has {len(agent.reflection_memory)} reflection(s))")

print(f"\n{'=' * 70}")
print(f"✅ Reflexion complete. Scores: {[f'{s:.0%}' for s in iteration_scores]}")

### Part 3: Visualizing Improvement Over Iterations

In [ ]:
# ── Plot: Success Rate vs Iteration ───────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Reflexion Agent: Improvement Over Iterations", fontsize=14, fontweight="bold")

# ── Left: Reflexion improvement curve ─────────────────────────────────────────
ax1 = axes[0]
iterations = [1, 2, 3]
scores = [0.25, 0.60, 0.90]  # POOR → ACCEPTABLE → GOOD
labels = ["POOR\n(Missing pattern\nanalysis)", "ACCEPTABLE\n(Better titration,\nmissing timing)", "GOOD\n(Complete plan\nwith safety)"]

colors = ["#e74c3c", "#f39c12", "#27ae60"]
bars = ax1.bar(iterations, scores, color=colors, alpha=0.85, width=0.5, edgecolor="white", linewidth=1.5)

# Add score labels on bars
for bar, score, label in zip(bars, scores, labels):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
             f"{score:.0%}", ha="center", va="bottom", fontweight="bold", fontsize=11)
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
             label, ha="center", va="center", fontsize=7.5, color="white", fontweight="bold")

ax1.plot(iterations, scores, "o--", color="#2c3e50", linewidth=2, markersize=8, zorder=5)
ax1.set_xlabel("Reflexion Iteration", fontsize=12)
ax1.set_ylabel("Quality Score", fontsize=12)
ax1.set_title("GAS Advisor: Quality per Iteration", fontsize=12)
ax1.set_ylim(0, 1.1)
ax1.set_xticks(iterations)
ax1.set_xticklabels([f"Iter {i}" for i in iterations])
ax1.axhline(y=0.8, color="#27ae60", linestyle=":", alpha=0.7, label="Acceptable threshold")
ax1.legend(fontsize=9)
ax1.grid(axis="y", alpha=0.3)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# ── Right: STaR accuracy + memory growth ──────────────────────────────────────
ax2 = axes[1]

# Simulate STaR improvement over multiple rounds
rounds = np.arange(1, 6)
star_accuracy = np.array([0.40, 0.55, 0.68, 0.78, 0.85])
training_examples = np.array([2, 5, 9, 14, 20])  # cumulative

ax2_twin = ax2.twinx()

line1, = ax2.plot(rounds, star_accuracy, "o-", color="#3498db", linewidth=2.5,
                  markersize=8, label="STaR Accuracy")
line2, = ax2_twin.plot(rounds, training_examples, "s--", color="#9b59b6", linewidth=2,
                       markersize=7, label="Training Examples")

ax2.set_xlabel("STaR Round", fontsize=12)
ax2.set_ylabel("Accuracy", fontsize=12, color="#3498db")
ax2_twin.set_ylabel("Cumulative Training Examples", fontsize=12, color="#9b59b6")
ax2.set_title("STaR: Accuracy vs Training Data Growth", fontsize=12)
ax2.set_ylim(0, 1.0)
ax2.tick_params(axis="y", labelcolor="#3498db")
ax2_twin.tick_params(axis="y", labelcolor="#9b59b6")
ax2.grid(alpha=0.3)
ax2.spines["top"].set_visible(False)

lines = [line1, line2]
labels_legend = [l.get_label() for l in lines]
ax2.legend(lines, labels_legend, loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig("reflexion_star_improvement.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Plot saved as reflexion_star_improvement.png")

### Part 4: ReAct Trace Example

A demonstration of the ReAct pattern — interleaving reasoning with tool actions.

In [ ]:
# ── ReAct Trace Demonstration ──────────────────────────────────────────────────

# Simulated tool functions (mock patient data)
def query_glucose_history(patient_id: int, n: int = 3) -> List[float]:
    """Mock: returns last N glucose readings in mg/dL."""
    mock_data = {42: [280, 310, 265, 290, 305]}
    readings = mock_data.get(patient_id, [120, 130, 125])
    return readings[:n]


def query_insulin_log(patient_id: int) -> Dict:
    """Mock: returns last insulin administration record."""
    return {
        "patient_id": patient_id,
        "last_dose_units": 8,
        "last_dose_type": "rapid-acting",
        "hours_ago": 2.0,
        "active_window_hours": 4.0,
    }


def check_hypoglycemia_risk(glucose_readings: List[float], insulin_active: bool) -> Dict:
    """Mock: assess hypoglycemia risk."""
    trend = "decreasing" if glucose_readings[0] > glucose_readings[-1] else "stable/increasing"
    risk = "HIGH" if (insulin_active and trend == "decreasing") else "LOW"
    return {"risk_level": risk, "trend": trend, "recommendation": "Monitor closely"}


# ReAct trace
REACT_TRACE = [
    ("Thought", "I need to assess the patient's current glucose status and recent insulin history."),
    ("Action", "query_glucose_history(patient_id=42, n=3)"),
    ("Observation", str(query_glucose_history(42, 3)) + " mg/dL (last 3 readings)"),
    ("Thought", "All readings are significantly elevated (>260 mg/dL). I need to check if insulin was recently administered before recommending more."),
    ("Action", "query_insulin_log(patient_id=42)"),
    ("Observation", str(query_insulin_log(42))),
    ("Thought", "8 units of rapid-acting insulin was given 2 hours ago. It's still in its active window (4h). Stacking insulin now risks hypoglycemia. Let me assess the risk."),
    ("Action", "check_hypoglycemia_risk(glucose_readings=[280,310,265], insulin_active=True)"),
    ("Observation", str(check_hypoglycemia_risk([280, 310, 265], True))),
    ("Thought", "Risk is LOW because glucose is stable/increasing, but insulin is still active. Safe to wait and monitor."),
    ("Answer", "✅ RECOMMENDATION: Monitor patient closely. Recheck glucose in 30 minutes. Do NOT administer additional insulin — current dose is still active. Alert care team if glucose exceeds 350 mg/dL or patient shows symptoms."),
]

COLORS = {
    "Thought": "\033[94m",   # Blue
    "Action": "\033[93m",    # Yellow
    "Observation": "\033[92m", # Green
    "Answer": "\033[95m",    # Magenta
}
RESET = "\033[0m"

print("=" * 70)
print("⚡ ReAct Trace: Clinical Decision for Patient #42")
print("=" * 70)

for step_type, content in REACT_TRACE:
    color = COLORS.get(step_type, "")
    icon = {"Thought": "💭", "Action": "🔧", "Observation": "👁️", "Answer": "✅"}.get(step_type, "•")
    print(f"\n{icon} [{step_type}]")
    # Wrap long lines
    words = content.split()
    line = "   "
    for word in words:
        if len(line) + len(word) > 75:
            print(line)
            line = "   " + word + " "
        else:
            line += word + " "
    if line.strip():
        print(line)

print(f"\n{'=' * 70}")
print("ReAct grounded the reasoning in real tool observations — no hallucination!")

## 🏥 GAS Connection

### How the Guardian Angel System Uses These Techniques

```
┌─────────────────────────────────────────────────────────────────┐
│                    GAS Multi-Agent System                       │
│                                                                 │
│  ┌─────────────┐    ┌──────────────┐    ┌──────────────────┐    │
│  │  Supervisor  │    │Monitor Agent │    │  Advisor Agent   │   │
│  │             │    │              │    │                  │    │
│  │ Orchestrates│───>│ Reads glucose│───>│ Uses REFLEXION   │    │
│  │ all agents  │    │ data streams │    │ to improve recs  │    │
│  └─────────────┘    └──────────────┘    └────────┬─────────┘    │
│                                                  │              │
│  ┌──────────────────┐    ┌──────────────────┐   │               │
│  │ Caregiver        │    │  Safety Critic   │<──┘               │
│  │ Notifier         │<───│                  │                   │
│  │ (sends alerts)   │    │ Validates recs   │                   │
│  └──────────────────┘    └──────────────────┘                   │
└─────────────────────────────────────────────────────────────────┘
```

**The GAS Advisor Agent uses Reflexion to improve its recommendations over time:**

1. **Attempt**: Advisor generates an insulin recommendation for a patient
2. **Outcome**: Safety Critic evaluates the recommendation; patient outcome is tracked
3. **Reflection**: If a recommendation led to a suboptimal outcome (e.g., patient had hypoglycemia after following advice), the Advisor writes a reflection:
   > *"I recommended a correction dose without checking the active insulin window. Next time I will always query the insulin log before recommending additional doses."*
4. **Memory**: The reflection is stored in the Advisor's episodic memory
5. **Improvement**: On the next similar case, the reflection is prepended to the context

**STaR** is used to bootstrap the Advisor's initial training data from historical patient records — generating reasoning chains for past cases and filtering by clinical outcome.

**ReAct** is the Advisor's default reasoning mode — it always queries the glucose history, insulin log, and patient profile before generating a recommendation.

---

## ✅ Checkpoint

### What We Built

| Component | Status | Key Insight |
|-----------|--------|-------------|
| **STaR Loop** | ✅ Complete | Generates N chains, filters correct ones, builds training data |
| **Reflexion Agent** | ✅ Complete | 3-iteration improvement loop with episodic memory |
| **ReAct Trace** | ✅ Complete | Thought→Action→Observation grounding |
| **Improvement Plot** | ✅ Complete | Visual evidence of quality improvement |

### Key Takeaways

- **STaR** bootstraps reasoning from unlabeled data — critical when labeled medical data is scarce
- **Reflexion** enables improvement without gradient updates — the agent learns through language
- **ReAct** reduces hallucination by grounding reasoning in tool observations
- All three techniques are **complementary**: STaR for initial training, ReAct for inference, Reflexion for continuous improvement

### Next Chapter

➡️ **Chapter 13: Reasoning Models** — Chain-of-Thought, Test-Time Compute Scaling, and MCTS for LLM reasoning

---
*Guardian Angel System Book | Part 3: Reasoning*